In [ ]:
pip install mcp httpx

In [ ]:
import os
import asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client
os.getenv('STITCH_API_KEY')

In [ ]:
import os
async def inspect_generation_tool():
    api_key = os.getenv('STITCH_API_KEY')
    client = StitchMCPClient(api_key)

    # Recuperamos la lista de herramientas
    res = await client._request("tools/list", {})
    tools = res.get('result', {}).get('tools', [])

    print("--- ESQUEMA DE GENERACIÓN DE PANTALLA ---")
    for t in tools:
        if t['name'] == 'generate_screen_from_text':
            print(f"Herramienta: {t['name']}")
            print(f"Input Schema: {json.dumps(t.get('inputSchema', {}), indent=2)}")

await inspect_generation_tool()

In [ ]:
import os
import httpx
import json
import traceback # Import traceback module

class StitchMCPClient:
    def __init__(self, api_key):
        self.url = "https://stitch.googleapis.com/mcp"
        self.headers = {"X-Goog-Api-Key": api_key, "Content-Type": "application/json"}
        self.request_id = 1

    async def _request(self, method, params):
        payload = {"jsonrpc": "2.0", "id": self.request_id, "method": method, "params": params}
        self.request_id += 1
        async with httpx.AsyncClient(timeout=120.0) as client: # Increased timeout to 120 seconds
            try:
                response = await client.post(self.url, headers=self.headers, json=payload)
                if response.status_code != 200:
                    print(f"[HTTP ERROR] {response.status_code}: {response.text}")
                    return {}
                return response.json()
            except Exception as e:
                print(f"[EXCEPTION] An error occurred during the request:")
                traceback.print_exc() # Print full traceback
                return {}

    async def call_tool(self, name, arguments):
        return await self._request("tools/call", {"name": name, "arguments": arguments})

async def run_complete_workflow():
    api_key = os.getenv('STITCH_API_KEY')
    if not api_key: return
    client = StitchMCPClient(api_key)

    print("1. Creando proyecto (usando 'title')...")
    res_proj = await client.call_tool("create_project", {"title": "Colab UI Project"})

    try:
        content_text = res_proj.get('result', {}).get('content', [{}])[0].get('text', '{}')
        project_data = json.loads(content_text)
        full_name = project_data.get('name', '') # format: 'projects/123'
        # Extraemos solo el ID numérico
        project_id = full_name.replace('projects/', '')
    except Exception as e:
        print("Error al obtener ID del proyecto:", res_proj)
        return

    if not project_id:
        print("Error: No se recibió project_id.")
        return

    print(f"OK. Clean Project ID: {project_id}")
    print("2. Generando pantalla...")

    # Corregido: 'projectId' es el parámetro requerido, sin el prefijo 'projects/'
    res_gen = await client.call_tool("generate_screen_from_text", {
        "projectId": project_id,
        "prompt": "A profesional fintech solution for compare prices of 2 prices of the same thing in the market",
        "deviceType": "DESKTOP",
        "modelId": "GEMINI_3_1_PRO"
    })

    print("\n--- RESULTADO DE GENERACIÓN ---")
    print(json.dumps(res_gen.get('result', {}), indent=2))

    # Extract and print the designMd (UI code)
    try:
        generated_content = json.loads(res_gen.get('result', {}).get('content', [{}])[0].get('text', '{}'))
        # Corrected path to designMd due to nested 'designSystem' keys
        design_md = generated_content.get('outputComponents', [{}])[0].get('designSystem', {}).get('designSystem', {}).get('designMd', 'No designMd found.')
        print("\n--- CÓDIGO DE LA UI GENERADA (Markdown) ---")
        print(design_md)
    except Exception as e:
        print(f"Error al extraer el código de la UI: {e}")

await run_complete_workflow()